# 08D — Promote the final segmentation configuration

## Objective

Notebook 08 selected the species-specific segmentation constants `C`.
Notebook 08C then showed that `f04` benefits from a small project-specific endpoint correction:

- `f04`: **20 ms symmetric padding per side**
- all other deployable species: **0 ms padding**

This notebook freezes that segmentation configuration for later field inference.

It does **not**:

- retrain GMMs;
- modify detector thresholds;
- use pond audio;
- retune `C`;
- change FIR bands, STE, MA=12, or the 3/3 endpoint rule.

The only promoted segmentation change is the reviewed `f04` padding.

In [1]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 100)

CWD = Path.cwd().resolve()
REPO_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

SEGMENTATION_DIR = (
    REPO_ROOT
    / "data"
    / "processed"
    / "candidate_call_segmentation"
)

CONFIG_PATH = (
    SEGMENTATION_DIR
    / "selected_segmentation_configuration.json"
)

DIAGNOSTIC_DIR = (
    SEGMENTATION_DIR
    / "f04_padding_end_to_end"
)

RECOMMENDATION_PATH = (
    DIAGNOSTIC_DIR
    / "f04_padding_recommendation.json"
)

SUMMARY_PATH = (
    DIAGNOSTIC_DIR
    / "f04_padding_end_to_end_summary.csv"
)

FINAL_CONFIG_PATH = (
    SEGMENTATION_DIR
    / "final_segmentation_configuration.json"
)

ARCHIVE_DIR = (
    SEGMENTATION_DIR
    / "archive"
)

ARCHIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for path in (
    CONFIG_PATH,
    RECOMMENDATION_PATH,
    SUMMARY_PATH,
):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required input: {path}"
        )

print("Segmentation dir :", SEGMENTATION_DIR)
print("Base config      :", CONFIG_PATH)
print("08C recommendation:", RECOMMENDATION_PATH)

Segmentation dir : C:\Users\sofia\Projects\PUCE\frog-presence-replication\data\processed\candidate_call_segmentation
Base config      : C:\Users\sofia\Projects\PUCE\frog-presence-replication\data\processed\candidate_call_segmentation\selected_segmentation_configuration.json
08C recommendation: C:\Users\sofia\Projects\PUCE\frog-presence-replication\data\processed\candidate_call_segmentation\f04_padding_end_to_end\f04_padding_recommendation.json


## 2. Load and verify the reviewed inputs

In [2]:
with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:
    base_config = json.load(f)

with open(
    RECOMMENDATION_PATH,
    "r",
    encoding="utf-8",
) as f:
    recommendation = json.load(f)

summary = pd.read_csv(
    SUMMARY_PATH
)

USABLE_CODES = tuple(
    base_config[
        "selection"
    ][
        "C_by_code"
    ].keys()
)

if recommendation[
    "target_code"
] != "f04":
    raise RuntimeError(
        "08D expects the 08C recommendation to target f04."
    )

if recommendation[
    "configuration_promoted"
] is not False:
    raise RuntimeError(
        "08C recommendation was expected to be diagnostic-only."
    )

if recommendation[
    "gmm_retrained"
] is not False:
    raise RuntimeError(
        "08C must not retrain GMMs."
    )

if recommendation[
    "thresholds_changed"
] is not False:
    raise RuntimeError(
        "08C must not change detector thresholds."
    )

if recommendation[
    "segmentation_C_changed"
] is not False:
    raise RuntimeError(
        "08C must not change segmentation C."
    )

if recommendation[
    "pond_audio_used"
] is not False:
    raise RuntimeError(
        "08C must not use pond audio."
    )

recommended_padding = int(
    recommendation[
        "recommended_padding_ms_per_side"
    ]
)

if recommended_padding != 20:
    raise RuntimeError(
        f"Expected reviewed f04 recommendation of 20 ms, "
        f"got {recommended_padding} ms."
    )

display(
    summary.round(4)
)

print("Reviewed f04 padding:", recommended_padding, "ms per side")

,padding_ms,n_candidates,candidate_seconds,valid_feature_fraction,segmentation_recall_50,accepted_f04_any_overlap_recall,end_to_end_recall_50,median_best_candidate_coverage,median_best_accepted_f04_coverage,candidate_acceptance_fraction,candidate_f04_acceptance_fraction,accepted_f04_candidates,accepted_f04_overlapping_annotation,accepted_f04_nonoverlap_annotation,raw_hypothesis_f04_fraction
0,0,2034,449.0700,1.0,0.7796,0.9301,0.7043,0.5465,0.5244,0.5787,0.5079,1033,178,855,0.6278
1,10,2033,489.7241,1.0,0.9462,0.9355,0.8763,0.5931,0.5750,0.5563,0.4934,1003,179,824,0.6257
2,20,1907,529.7341,1.0,0.9946,0.9301,0.9247,0.6429,0.6250,0.5128,0.4646,886,175,711,0.6203


Reviewed f04 padding: 20 ms per side


## 3. Build the frozen padding map

Only `f04` receives padding.

In [3]:
PADDING_MS_BY_CODE = {
    code: 0
    for code in USABLE_CODES
}

PADDING_MS_BY_CODE[
    "f04"
] = recommended_padding

padding_table = pd.DataFrame([
    {
        "code": code,
        "C": float(
            base_config[
                "selection"
            ][
                "C_by_code"
            ][
                code
            ]
        ),
        "padding_ms_per_side": int(
            PADDING_MS_BY_CODE[
                code
            ]
        ),
    }
    for code in USABLE_CODES
])

display(
    padding_table
)

,code,C,padding_ms_per_side
0,f01,4.00,0
1,f02,2.00,0
2,f03,4.00,0
3,f04,10.00,20
4,f05,2.50,0
5,f06,4.00,0
6,f07,4.00,0
7,f08,1.75,0
8,f10,4.00,0


## 4. Archive the pre-padding segmentation configuration

The notebook-08 configuration is kept unchanged for provenance.

In [4]:
ARCHIVE_PATH = (
    ARCHIVE_DIR
    / "selected_segmentation_configuration_pre_08C_padding.json"
)

if not ARCHIVE_PATH.exists():
    shutil.copy2(
        CONFIG_PATH,
        ARCHIVE_PATH,
    )

print("Archived base configuration:", ARCHIVE_PATH)

Archived base configuration: C:\Users\sofia\Projects\PUCE\frog-presence-replication\data\processed\candidate_call_segmentation\archive\selected_segmentation_configuration_pre_08C_padding.json


## 5. Create the final frozen segmentation configuration

In [5]:
final_config = dict(
    base_config
)

final_config[
    "status"
] = "final_frozen_after_08C_padding_review"

final_config[
    "endpoint_padding_ms_per_side"
] = {
    code: int(
        PADDING_MS_BY_CODE[
            code
        ]
    )
    for code in USABLE_CODES
}

final_config[
    "padding_provenance"
] = {
    "source_notebook":
        "08C_f04_padding_end_to_end",
    "target_code":
        "f04",
    "selected_padding_ms_per_side":
        recommended_padding,
    "selection_basis":
        (
            "maximize strict end-to-end f04 call recovery "
            "with >=50% manual-call coverage through the frozen detector"
        ),
    "paper_status":
        (
            "project-specific extension; endpoint padding is not reported "
            "in the source paper"
        ),
    "pond_audio_used_for_selection":
        False,
}

final_config[
    "frozen_for_field_inference"
] = True

final_config[
    "field_inference_rule"
] = (
    "apply species-specific FIR/STE segmentation using frozen C, "
    "then symmetric endpoint padding from endpoint_padding_ms_per_side; "
    "extract PLP-RASTA from original unfiltered candidate waveform"
)

final_config[
    "do_not_retune_on_ponds"
] = True

with open(
    FINAL_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_config,
        f,
        indent=2,
    )

print("Final segmentation configuration:", FINAL_CONFIG_PATH)

Final segmentation configuration: C:\Users\sofia\Projects\PUCE\frog-presence-replication\data\processed\candidate_call_segmentation\final_segmentation_configuration.json


## 6. Final safety checks

In [6]:
with open(
    FINAL_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:
    loaded_final = json.load(f)

assert (
    loaded_final[
        "status"
    ]
    == "final_frozen_after_08C_padding_review"
)

assert (
    loaded_final[
        "frozen_for_field_inference"
    ]
    is True
)

assert (
    loaded_final[
        "do_not_retune_on_ponds"
    ]
    is True
)

assert (
    loaded_final[
        "endpoint_padding_ms_per_side"
    ][
        "f04"
    ]
    == 20
)

for code in USABLE_CODES:
    if code == "f04":
        continue

    assert (
        loaded_final[
            "endpoint_padding_ms_per_side"
        ][
            code
        ]
        == 0
    )

assert (
    loaded_final[
        "selection"
    ][
        "C_by_code"
    ]
    == base_config[
        "selection"
    ][
        "C_by_code"
    ]
)

print("f04 padding                 : 20 ms per side")
print("Other species padding       : 0 ms")
print("Species-specific C values   : unchanged")
print("FIR / STE / MA / 3-3 logic : unchanged")
print("Pond audio used             : NO")
print("Segmentation configuration  : FROZEN")

f04 padding                 : 20 ms per side
Other species padding       : 0 ms
Species-specific C values   : unchanged
FIR / STE / MA / 3-3 logic : unchanged
Pond audio used             : NO
Segmentation configuration  : FROZEN


## 7. Final segmentation manifest

After this notebook finishes successfully, the segmentation frontend is frozen for notebook 09.

Operationally:

1. process long audio in **30 s** blocks;
2. apply the species-specific FIR band for segmentation only;
3. compute **10 ms non-overlapping STE**;
4. smooth with **MA=12**;
5. use the frozen species-specific `C`;
6. use **3 consecutive above / 3 consecutive below** endpoint logic;
7. add **20 ms per side only for f04**;
8. clip padded endpoints to the source WAV boundaries;
9. extract PLP-RASTA from the corresponding **original unfiltered waveform**;
10. pass candidates to the frozen final GMM + LLR detector.

No additional segmentation tuning should happen on pond recordings unless a separate, explicitly
declared development subset is introduced. Otherwise the final evaluation would become a charming
exercise in grading our own homework.